In [1]:
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
import pickle
import json
import os
from constants import *
from utils_download import *

username_token = ''
date = '20241205'
lb = 'new'

In [2]:
#outputs = []
#for model in tqdm(models):
#    outputs.append(clone_repo(f'https://{username_token}@huggingface.co/datasets/{model}/', target_directory=f'new_repos/{model}'.replace("open-llm-leaderboard/","")))

In [3]:
models = MODELS
scenarios = SCENARIOS

In [4]:
data = {}
for model in tqdm(models):
    data[model] = {}
    for s in scenarios:
        scenario = s.split('|')[0]
        data[model][scenario] = {}
        
        base_dir = "new_repos/"+model.replace("open-llm-leaderboard/","")+"/"+model.replace("open-llm-leaderboard/","").replace("-details","")+"/"
        
        if os.path.isdir(base_dir):
            file = find_folder_with_file_new(s, base_dir) 
        else:
            file = None
        
        if file is not None: 
            file = find_folder_with_file_new(s, base_dir)
            with open(base_dir+file, 'r') as json_file:
                aux = list(json_file)
                
            if 'ifeval' in scenario:
                aux1 = np.array([np.mean(json.loads(a)['inst_level_strict_acc']) for a in aux])
                weight1 = np.array([len(json.loads(a)['inst_level_strict_acc']) for a in aux]).astype(float)
                weight1 /= weight1.mean()
                aux2 = np.array([np.mean(json.loads(a)['prompt_level_strict_acc']) for a in aux])
                data[model][scenario]['correctness'] = {'correctness_inst':aux1, 'weight_inst': weight1,
                                                        'correctness_prompt':aux2, 'weight_prompt': np.ones(len(aux2)).astype(float)}
            else:
                if 'mmlu_pro' in scenario: metric = 'acc'
                elif 'math' in scenario: metric = 'exact_match'
                else: metric = 'acc_norm'
                data[model][scenario]['correctness'] = np.array([json.loads(a)[metric] for a in aux])

        else:
            data[model][scenario]['dates'] = None
            data[model][scenario]['correctness'] = None

with open(f'data/new_leaderboard_raw_{date}.pickle', 'wb') as handle:
    pickle.dump(data, handle, protocol=pickle.HIGHEST_PROTOCOL)

  0%|          | 0/20 [00:00<?, ?it/s]

In [5]:
def conversion(s):
    if isinstance(s, (int, float)):
        return s
    else:
        try:
            return float(s)
        except:
            return {'1':1,'0':0,'1.0':1,'0.0':0,'True':1,'False':0, True:1, False:0}[s]

with open(f'data/{lb}_leaderboard_raw_{date}.pickle', 'rb') as handle:
    data = pickle.load(handle)

pdata = {}
for s in tqdm(np.unique(sum([list(data[key].keys()) for key in data.keys()], [])).tolist()):
    if 'ifeval' in s:
        pdata[s] = {}
        pdata[s]['correctness'] = {}

        for correct in ['inst', 'prompt']:
            ###
            aux_data = [data[m][s]['correctness']['correctness_'+correct] for m in data.keys() if data[m][s] is not None and data[m][s]['correctness'] is not None]
            aux_data = [[conversion(s) for s in a] for a in  aux_data]
            aux_models = [m for m in data.keys() if data[m][s] is not None and data[m][s]['correctness'] is not None]
            valid = np.array([np.array(a).shape[0] for a in aux_data])
            valid = valid==np.median(valid)
            aux_data = [m for i,m in enumerate(aux_data) if valid[i]] 
            aux_models = [m for i,m in enumerate(aux_models) if valid[i]] 
            
            ###
            pdata[s]['correctness']['correctness_'+correct] = np.array(aux_data).astype(float)
            pdata[s]['correctness']['weights_'+correct] = data[aux_models[0]][s]['correctness']['weight_'+correct]
            pdata[s]['models'] = aux_models
    else:
        ###
        aux_data = [data[m][s]['correctness'] for m in data.keys() if data[m][s] is not None and data[m][s]['correctness'] is not None]
        aux_data = [[conversion(s) for s in a] for a in  aux_data]
        aux_models = [m for m in data.keys() if data[m][s] is not None and data[m][s]['correctness'] is not None]
        valid = np.array([np.array(a).shape[0] for a in aux_data])
        valid = valid==np.median(valid)
        aux_data = [m for i,m in enumerate(aux_data) if valid[i]] 
        aux_models = [m for i,m in enumerate(aux_models) if valid[i]] 
        
        ###
        pdata[s] = {}
        pdata[s]['correctness'] = np.array(aux_data).astype(float)
        pdata[s]['models'] = aux_models

with open(f'data/{lb}_leaderboard_processed_{date}.pickle', 'wb') as handle:
    pickle.dump(pdata, handle, protocol=pickle.HIGHEST_PROTOCOL)

  0%|          | 0/39 [00:00<?, ?it/s]